# AI-Based Smart Route Optimization System

## Project Theme
This notebook designs and tests an intelligent route optimization system using multiple Artificial Intelligence algorithms.

## Algorithms Implemented
1. **Breadth First Search (BFS)**
2. **Depth First Search (DFS)**
3. **A\* Search**
4. **Hill Climbing**

## Main Objective
The system finds a route between a current location and a destination, then compares different AI algorithms using:

- Total distance
- Traffic score
- Estimated travel time
- Total route cost
- Number of expanded nodes
- Runtime in milliseconds
- Whether the destination was reached

This makes the project suitable for explaining search, heuristic search, greedy optimization, and algorithm comparison.

## Step 1: Import Required Libraries

We use standard Python libraries for algorithms and data handling.

- `heapq` is used for the priority queue in A\* Search.
- `deque` is used for BFS.
- `pandas` is used to build comparison tables.
- `networkx` and `matplotlib` are used for graph visualization.
- `time` is used to measure algorithm runtime.

In [ ]:
import math
import time
import heapq
from dataclasses import dataclass
from collections import deque
from typing import Dict, List, Tuple, Optional

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option("display.max_colwidth", 120)

## Step 2: Define the Data Model

A good project should not only work, but should also have clean structure.

So instead of using random lists everywhere, we define:

- `Location`: stores a city point with coordinates.
- `Edge`: stores a route between two locations with distance and traffic.
- `SearchResult`: stores the final result of every algorithm.

This makes the project easier to explain and more professional.

In [ ]:
@dataclass
class Location:
    name: str
    x: float
    y: float
    description: str


@dataclass
class Edge:
    destination: str
    distance: float
    traffic: int


@dataclass
class SearchResult:
    algorithm: str
    path: List[str]
    distance: float
    traffic: int
    estimated_time: float
    total_cost: float
    expanded_nodes: int
    runtime_ms: float
    reached_goal: bool

## Step 3: Create the City Map Dataset

The city is represented as a graph.

- Each **node** represents a location.
- Each **edge** represents a road between two locations.
- Every road has:
  - Distance in kilometers
  - Traffic score
  - Estimated time calculated from distance and traffic

This graph is small enough to explain in viva, but detailed enough to demonstrate real AI search behavior.

In [ ]:
locations: Dict[str, Location] = {
    "Home": Location("Home", 0, 4, "Residential starting area"),
    "Market": Location("Market", 2, 6, "Commercial market area"),
    "School": Location("School", 3, 3, "Educational area"),
    "Hospital": Location("Hospital", 5, 5, "Medical emergency facility"),
    "Mall": Location("Mall", 6, 7, "Shopping center"),
    "Park": Location("Park", 7, 3, "Public park and open area"),
    "University": Location("University", 9, 5, "University campus"),
    "Bus Stop": Location("Bus Stop", 4, 1, "Public transport point"),
    "Airport": Location("Airport", 10, 8, "Airport route"),
    "Office": Location("Office", 12, 4, "Office destination area"),
}

graph: Dict[str, List[Edge]] = {
    "Home": [
        Edge("Market", 4, 2),
        Edge("School", 2, 1),
        Edge("Bus Stop", 5, 3),
    ],
    "Market": [
        Edge("Home", 4, 2),
        Edge("Hospital", 5, 3),
        Edge("Mall", 6, 4),
    ],
    "School": [
        Edge("Home", 2, 1),
        Edge("Hospital", 3, 2),
        Edge("Bus Stop", 4, 1),
        Edge("Park", 6, 2),
    ],
    "Hospital": [
        Edge("Market", 5, 3),
        Edge("School", 3, 2),
        Edge("University", 4, 2),
        Edge("Park", 3, 1),
    ],
    "Mall": [
        Edge("Market", 6, 4),
        Edge("Airport", 5, 3),
        Edge("University", 4, 2),
    ],
    "Park": [
        Edge("Hospital", 3, 1),
        Edge("University", 3, 1),
        Edge("Bus Stop", 5, 2),
        Edge("School", 6, 2),
    ],
    "University": [
        Edge("Hospital", 4, 2),
        Edge("Mall", 4, 2),
        Edge("Park", 3, 1),
        Edge("Office", 2, 1),
    ],
    "Bus Stop": [
        Edge("Home", 5, 3),
        Edge("School", 4, 1),
        Edge("Park", 5, 2),
        Edge("Airport", 9, 4),
    ],
    "Airport": [
        Edge("Mall", 5, 3),
        Edge("Office", 6, 4),
        Edge("Bus Stop", 9, 4),
    ],
    "Office": [
        Edge("University", 2, 1),
        Edge("Airport", 6, 4),
    ],
}

print("Total locations:", len(locations))
print("Total directed road connections:", sum(len(edges) for edges in graph.values()))

## Step 4: Utility Functions

These helper functions keep the main algorithms clean.

### Heuristic Function
For A\* and Hill Climbing, we use Euclidean distance as a heuristic.

The heuristic estimates how close a location is to the goal.

\[
h(n) = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}
\]

### Route Priority
The user can optimize the route according to:

- Shortest Distance
- Least Traffic
- Fastest Time
- Balanced Route

In [ ]:
def heuristic(node: str, goal: str) -> float:
    # Euclidean distance used as a heuristic estimate from current node to goal.
    n1 = locations[node]
    n2 = locations[goal]
    return math.sqrt((n2.x - n1.x) ** 2 + (n2.y - n1.y) ** 2)


def estimate_time(distance: float, traffic: int) -> float:
    # Traffic increases travel time.
    return distance * 4 + traffic * 3


def edge_priority_cost(edge: Edge, priority: str) -> float:
    # Returns edge cost according to selected user priority.
    time_value = estimate_time(edge.distance, edge.traffic)

    if priority == "Shortest Distance":
        return edge.distance

    if priority == "Least Traffic":
        return edge.distance + edge.traffic * 3

    if priority == "Fastest Time":
        return time_value

    # Balanced route considers distance, traffic, and time together.
    return edge.distance + edge.traffic + time_value / 10


def get_edge(source: str, destination: str) -> Optional[Edge]:
    # Find edge data between two connected nodes.
    for edge in graph[source]:
        if edge.destination == destination:
            return edge
    return None


def calculate_path_metrics(path: List[str]) -> Tuple[float, int, float, float]:
    # Calculates distance, traffic, estimated time, and total cost for a route.
    if not path or len(path) < 2:
        return 0, 0, 0, 0

    total_distance = 0
    total_traffic = 0
    total_time = 0

    for i in range(len(path) - 1):
        edge = get_edge(path[i], path[i + 1])

        if edge is not None:
            total_distance += edge.distance
            total_traffic += edge.traffic
            total_time += estimate_time(edge.distance, edge.traffic)

    total_cost = total_distance + total_traffic + total_time / 10

    return total_distance, total_traffic, total_time, total_cost


def build_result(
    algorithm: str,
    path: List[str],
    goal: str,
    expanded_nodes: int,
    runtime_ms: float
) -> SearchResult:
    # Creates a SearchResult object for a specific algorithm.
    distance, traffic, estimated_time_value, total_cost = calculate_path_metrics(path)
    reached_goal = bool(path) and path[-1] == goal

    return SearchResult(
        algorithm=algorithm,
        path=path,
        distance=distance,
        traffic=traffic,
        estimated_time=estimated_time_value,
        total_cost=total_cost,
        expanded_nodes=expanded_nodes,
        runtime_ms=runtime_ms,
        reached_goal=reached_goal,
    )

# Step 5: Implement AI Algorithms

## 5.1 Breadth First Search

BFS explores the graph level by level.

### Why BFS is useful?
It can find the path with the minimum number of edges if all edges are treated equally.

### Limitation
It does not care about distance or traffic unless modified.

In [ ]:
def bfs(start: str, goal: str) -> Tuple[List[str], int]:
    queue = deque([[start]])
    visited = set()
    expanded_nodes = 0

    while queue:
        path = queue.popleft()
        current_node = path[-1]

        if current_node == goal:
            return path, expanded_nodes

        if current_node not in visited:
            visited.add(current_node)
            expanded_nodes += 1

            for edge in graph[current_node]:
                if edge.destination not in visited:
                    queue.append(path + [edge.destination])

    return [], expanded_nodes

## 5.2 Depth First Search

DFS explores one path deeply before backtracking.

### Why DFS is useful?
It is simple and uses less memory in many cases.

### Limitation
It may not find the shortest or most efficient path.

In [ ]:
def dfs(start: str, goal: str) -> Tuple[List[str], int]:
    stack = [[start]]
    visited = set()
    expanded_nodes = 0

    while stack:
        path = stack.pop()
        current_node = path[-1]

        if current_node == goal:
            return path, expanded_nodes

        if current_node not in visited:
            visited.add(current_node)
            expanded_nodes += 1

            for edge in graph[current_node]:
                if edge.destination not in visited:
                    stack.append(path + [edge.destination])

    return [], expanded_nodes

## 5.3 A\* Search

A\* is an informed search algorithm.

It uses:

\[
f(n) = g(n) + h(n)
\]

Where:

- `g(n)` = actual path cost from start to current node
- `h(n)` = estimated cost from current node to destination
- `f(n)` = final priority value

### Why A\* is powerful?
It combines actual cost and heuristic estimation, so it usually finds an efficient route faster than blind search.

In [ ]:
def astar(start: str, goal: str, priority: str) -> Tuple[List[str], int]:
    priority_queue = []
    heapq.heappush(priority_queue, (0, 0, start, [start]))

    best_cost = {}
    expanded_nodes = 0

    while priority_queue:
        f_cost, g_cost, current_node, path = heapq.heappop(priority_queue)

        if current_node == goal:
            return path, expanded_nodes

        if current_node in best_cost and best_cost[current_node] <= g_cost:
            continue

        best_cost[current_node] = g_cost
        expanded_nodes += 1

        for edge in graph[current_node]:
            new_g_cost = g_cost + edge_priority_cost(edge, priority)
            new_f_cost = new_g_cost + heuristic(edge.destination, goal)
            new_path = path + [edge.destination]

            heapq.heappush(
                priority_queue,
                (new_f_cost, new_g_cost, edge.destination, new_path)
            )

    return [], expanded_nodes

## 5.4 Hill Climbing

Hill Climbing is a greedy optimization technique.

At every step, it chooses the neighbor that looks closest to the destination.

### Why Hill Climbing is useful?
It is fast and easy to understand.

### Limitation
It can get stuck at a local optimum and may fail to reach the destination.

In [ ]:
def hill_climbing(start: str, goal: str, priority: str) -> Tuple[List[str], int]:
    current_node = start
    path = [current_node]
    visited = set()
    expanded_nodes = 0

    while current_node != goal:
        visited.add(current_node)
        expanded_nodes += 1

        candidate_neighbors = []

        for edge in graph[current_node]:
            if edge.destination not in visited:
                score = (
                    heuristic(edge.destination, goal)
                    + edge_priority_cost(edge, priority) * 0.20
                )
                candidate_neighbors.append((score, edge.destination))

        if not candidate_neighbors:
            return path, expanded_nodes

        candidate_neighbors.sort()
        current_node = candidate_neighbors[0][1]
        path.append(current_node)

        if len(path) > len(graph):
            break

    return path, expanded_nodes

## Step 6: Run Experiments

This function runs all four algorithms on the same start and destination.

It also measures runtime for each algorithm, making the comparison more professional.

In [ ]:
def run_algorithm(algorithm_name: str, start: str, goal: str, priority: str) -> SearchResult:
    start_time = time.perf_counter()

    if algorithm_name == "BFS":
        path, expanded_nodes = bfs(start, goal)

    elif algorithm_name == "DFS":
        path, expanded_nodes = dfs(start, goal)

    elif algorithm_name == "A* Search":
        path, expanded_nodes = astar(start, goal, priority)

    elif algorithm_name == "Hill Climbing":
        path, expanded_nodes = hill_climbing(start, goal, priority)

    else:
        raise ValueError("Unknown algorithm name")

    end_time = time.perf_counter()
    runtime_ms = (end_time - start_time) * 1000

    return build_result(
        algorithm=algorithm_name,
        path=path,
        goal=goal,
        expanded_nodes=expanded_nodes,
        runtime_ms=runtime_ms
    )


def run_all_algorithms(start: str, goal: str, priority: str) -> pd.DataFrame:
    algorithms = ["BFS", "DFS", "A* Search", "Hill Climbing"]
    results = []

    for algorithm in algorithms:
        result = run_algorithm(algorithm, start, goal, priority)
        results.append({
            "Algorithm": result.algorithm,
            "Route": " → ".join(result.path) if result.path else "No path found",
            "Distance (km)": result.distance,
            "Traffic Score": result.traffic,
            "Estimated Time (min)": result.estimated_time,
            "Total Cost": round(result.total_cost, 2),
            "Expanded Nodes": result.expanded_nodes,
            "Runtime (ms)": round(result.runtime_ms, 5),
            "Reached Goal": "Yes" if result.reached_goal else "No"
        })

    return pd.DataFrame(results)

## Step 7: Choose Start, Destination, and Priority

Change only these three variables to test different cases.

In [ ]:
START_LOCATION = "Home"
DESTINATION_LOCATION = "Office"

# Choose one:
# "Shortest Distance"
# "Least Traffic"
# "Fastest Time"
# "Balanced Route"
ROUTE_PRIORITY = "Shortest Distance"

results_df = run_all_algorithms(
    start=START_LOCATION,
    goal=DESTINATION_LOCATION,
    priority=ROUTE_PRIORITY
)

print("Start Location:", START_LOCATION)
print("Destination:", DESTINATION_LOCATION)
print("Selected Priority:", ROUTE_PRIORITY)

display(results_df)

## Step 8: Select the Best Algorithm

The best route is selected according to the user's selected priority.

For example:

- If priority is `Shortest Distance`, the route with minimum distance is selected.
- If priority is `Least Traffic`, the route with minimum traffic score is selected.
- If priority is `Fastest Time`, the route with minimum estimated time is selected.
- If priority is `Balanced Route`, total cost is considered.

In [ ]:
def find_best_result(results: pd.DataFrame, priority: str) -> pd.Series:
    valid_results = results[results["Reached Goal"] == "Yes"].copy()

    if valid_results.empty:
        raise ValueError("No algorithm reached the destination.")

    if priority == "Shortest Distance":
        return valid_results.loc[valid_results["Distance (km)"].idxmin()]

    if priority == "Least Traffic":
        return valid_results.loc[valid_results["Traffic Score"].idxmin()]

    if priority == "Fastest Time":
        return valid_results.loc[valid_results["Estimated Time (min)"].idxmin()]

    return valid_results.loc[valid_results["Total Cost"].idxmin()]


best_result = find_best_result(results_df, ROUTE_PRIORITY)

print("Best Algorithm:", best_result["Algorithm"])
print("Best Route:", best_result["Route"])
print("Distance:", best_result["Distance (km)"], "km")
print("Traffic Score:", best_result["Traffic Score"])
print("Estimated Time:", best_result["Estimated Time (min)"], "minutes")

## Step 9: Graph Visualization

This function draws the city graph and highlights the selected algorithm path.

- Blue nodes represent locations.
- Gray lines represent all possible roads.
- Green nodes and red route represent the selected path.

In [ ]:
def create_networkx_graph() -> nx.Graph:
    G = nx.Graph()

    for node_name, location in locations.items():
        G.add_node(node_name, pos=(location.x, location.y))

    added_edges = set()

    for source, edges in graph.items():
        for edge in edges:
            edge_key = tuple(sorted([source, edge.destination]))

            if edge_key not in added_edges:
                added_edges.add(edge_key)
                G.add_edge(
                    source,
                    edge.destination,
                    distance=edge.distance,
                    traffic=edge.traffic
                )

    return G


def draw_city_graph(path: Optional[List[str]] = None, title: str = "City Route Graph") -> None:
    G = create_networkx_graph()
    pos = nx.get_node_attributes(G, "pos")

    plt.figure(figsize=(13, 7))

    nx.draw_networkx_edges(
        G,
        pos,
        edge_color="gray",
        width=2,
        alpha=0.6
    )

    nx.draw_networkx_nodes(
        G,
        pos,
        node_color="lightblue",
        node_size=2300,
        edgecolors="black"
    )

    nx.draw_networkx_labels(
        G,
        pos,
        font_size=9,
        font_weight="bold"
    )

    edge_labels = {
        (u, v): f"{data['distance']}km | T:{data['traffic']}"
        for u, v, data in G.edges(data=True)
    }

    nx.draw_networkx_edge_labels(
        G,
        pos,
        edge_labels=edge_labels,
        font_size=8
    )

    if path and len(path) > 1:
        route_edges = list(zip(path, path[1:]))

        nx.draw_networkx_edges(
            G,
            pos,
            edgelist=route_edges,
            edge_color="red",
            width=5
        )

        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=path,
            node_color="lightgreen",
            node_size=2500,
            edgecolors="black"
        )

    plt.title(title, fontsize=16, fontweight="bold")
    plt.axis("off")
    plt.show()

## Step 10: Show All Algorithm Routes

In [ ]:
draw_city_graph(title="Complete City Road Network")

for _, row in results_df.iterrows():
    route_text = row["Route"]

    if route_text != "No path found":
        route_path = route_text.split(" → ")
        draw_city_graph(
            path=route_path,
            title=f"{row['Algorithm']} Route"
        )

## Step 11: Performance Comparison Charts

These charts help explain algorithm performance visually.

They compare:

- Distance
- Traffic score
- Estimated time
- Expanded nodes
- Runtime

In [ ]:
def plot_comparison_bar(results: pd.DataFrame, column: str, title: str, ylabel: str) -> None:
    plt.figure(figsize=(9, 5))
    plt.bar(results["Algorithm"], results[column])
    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel("Algorithm")
    plt.ylabel(ylabel)
    plt.grid(axis="y", alpha=0.3)
    plt.show()


plot_comparison_bar(results_df, "Distance (km)", "Distance Comparison", "Distance in km")
plot_comparison_bar(results_df, "Traffic Score", "Traffic Score Comparison", "Traffic Score")
plot_comparison_bar(results_df, "Estimated Time (min)", "Estimated Time Comparison", "Time in minutes")
plot_comparison_bar(results_df, "Expanded Nodes", "Search Efficiency Comparison", "Expanded Nodes")
plot_comparison_bar(results_df, "Runtime (ms)", "Runtime Comparison", "Runtime in milliseconds")

## Step 12: Interactive Testing in Colab

Run this cell and select:

- Start location
- Destination
- Route priority

Then click **Run Route Optimization**.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    start_dropdown = widgets.Dropdown(
        options=list(locations.keys()),
        value="Home",
        description="Start:"
    )

    goal_dropdown = widgets.Dropdown(
        options=list(locations.keys()),
        value="Office",
        description="Goal:"
    )

    priority_dropdown = widgets.Dropdown(
        options=[
            "Shortest Distance",
            "Least Traffic",
            "Fastest Time",
            "Balanced Route"
        ],
        value="Shortest Distance",
        description="Priority:"
    )

    run_button = widgets.Button(
        description="Run Route Optimization",
        button_style="success"
    )

    output_area = widgets.Output()

    def on_run_button_clicked(button):
        with output_area:
            clear_output()

            start_value = start_dropdown.value
            goal_value = goal_dropdown.value
            priority_value = priority_dropdown.value

            if start_value == goal_value:
                print("Start and destination cannot be the same.")
                return

            temp_results = run_all_algorithms(start_value, goal_value, priority_value)
            display(temp_results)

            temp_best = find_best_result(temp_results, priority_value)

            print("\nBest Algorithm:", temp_best["Algorithm"])
            print("Best Route:", temp_best["Route"])

            best_path = temp_best["Route"].split(" → ")
            draw_city_graph(best_path, title="Best Route Selected by the System")

    run_button.on_click(on_run_button_clicked)

    display(start_dropdown, goal_dropdown, priority_dropdown, run_button, output_area)

except Exception as error:
    print("Interactive widgets could not be loaded.")
    print("Error:", error)

# Final Explanation for Instructor

## What this project demonstrates

This project demonstrates how AI search and optimization techniques can solve a real-world route planning problem.

## Why multiple algorithms were used

Different algorithms solve the same problem differently:

- **BFS** explores level by level.
- **DFS** explores deeply.
- **A\*** uses cost and heuristic to make smarter decisions.
- **Hill Climbing** greedily chooses the next best location.

## Best algorithm for this project

A\* Search is generally the best choice because it uses both actual cost and heuristic distance.  
This makes it more intelligent than BFS and DFS.

## Limitation

The current system uses a fixed city graph instead of real-time Google Maps data.  
In future, it can be improved by adding real traffic API data and GPS-based location selection.

## Future Enhancements

- Add real-time maps
- Add live traffic
- Add accident or road blockage detection
- Add fuel cost calculation
- Add emergency route mode for ambulance or rescue vehicles